In [2]:
"""
Ontario IESO hourly electricity consumption by FSA  ->  48 split CSV files.

HOW TO USE
    1. pip install pandas requests holidays
    2. Edit the CONFIG block below (months, output folder).
    3. Just run the file (python build_dataset.py, or Run in your IDE / notebook).

Output folder will contain only:
    Business_Day_0100.csv ... Business_Day_2400.csv
    Not_Business_Day_0100.csv ... Not_Business_Day_2400.csv
"""
import io
import re
import tempfile
import time
import zipfile
from pathlib import Path

import holidays
import pandas as pd
import requests

# =============================== CONFIG ===================================== #
START_YEAR, START_MONTH = 2022, 1
END_YEAR, END_MONTH = 2023, 12            # inclusive

OUTPUT_DIR = "train_data"                 # e.g. "train_data" / "test_data"

# Optional extras (leave as None for now)
CENTROIDS_CSV = None      # CSV with columns FSA, LAT, LON  -> adds LAT/LON for heat maps
TEMPERATURE_CSV = None    # CSV with FSA, DATE, HOUR, TEMPERATURE (+ optional weather cols)

KEEP_PRICE_PLANS = {"TIERED", "TOU"}
LOCAL_ZIP_DIR = None      # set to a folder of already-downloaded zips to skip downloading
PRINT_DISTINCT_VALUES = True   # prints PRICE_PLAN / CUSTOMER_TYPE values once, as a sanity check
# ============================================================================ #

BASE_URL = "https://reports-public.ieso.ca/public/HourlyConsumptionByFSA"
FSA_RE = re.compile(r"^[A-Z]\d[A-Z]$")      # letter-digit-letter only (drops 'K1', 'M5' style)

# canonical name -> possible header names in the IESO file (after normalising)
COLUMN_CANDIDATES = {
    "FSA": ["FSA", "FORWARD_SORTATION_AREA"],
    "DATE": ["DATE", "DELIVERY_DATE"],
    "HOUR": ["HOUR", "DELIVERY_HOUR"],
    "CUSTOMER_TYPE": ["CUSTOMER_TYPE", "CUSTOMERTYPE", "CUSTOMER_CLASS"],
    "PRICE_PLAN": ["PRICE_PLAN", "PRICEPLAN", "PRICE_PLAN_TYPE"],
    "TOTAL_CONSUMPTION": ["TOTAL_CONSUMPTION", "TOTAL_CONSUMPTION_KWH", "CONSUMPTION_KWH"],
    "PREMISE_COUNT": ["PREMISE_COUNT", "PREMISES", "NUMBER_OF_PREMISES"],
}

# Output column order. Columns that don't exist (e.g. no centroid/temperature file) are skipped.
OUTPUT_ORDER = [
    "DATE", "HOUR", "FSA", "CUSTOMER_TYPE", "PRICE_PLAN",
    "TOTAL_CONSUMPTION", "PREMISE_COUNT", "CONSUMPTION_PER_UNIT",
    "IS_WEEKDAY", "IS_HOLIDAY", "DAY_NAME", "YEAR", "MONTH", "WEEK_OF_YEAR", "DAY_OF_YEAR",
    "LAT", "LON", "REGION_LETTER", "IS_RURAL",
    "TEMPERATURE", "FEELS_LIKE", "HUMIDITY", "WIND_SPEED", "SOLAR_RADIATION", "CLOUD_COVER",
    "HDD", "CDD",
]
WEATHER_COLS = ["TEMPERATURE", "FEELS_LIKE", "HUMIDITY", "WIND_SPEED", "SOLAR_RADIATION", "CLOUD_COVER"]
BASE_TEMP_C = 18.0


# ------------------------------- helpers ------------------------------------ #
def norm_col(c):
    return re.sub(r"[^A-Z0-9]+", "_", str(c).strip().upper()).strip("_")


def month_range(sy, sm, ey, em):
    y, m = sy, sm
    while (y, m) <= (ey, em):
        yield y, m
        m += 1
        if m == 13:
            y, m = y + 1, 1


def download_zip(year, month, dest_dir, retries=3):
    name = f"PUB_HourlyConsumptionByFSA_{year}{month:02d}_v1.zip"
    path = Path(dest_dir) / name
    if path.exists():
        return path
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(f"{BASE_URL}/{name}", stream=True, timeout=120)
            if r.status_code == 404:
                print(f"  [skip] {name} is not published")
                return None
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
            return path
        except requests.RequestException as e:
            print(f"  [retry {attempt}/{retries}] {e}")
            time.sleep(2 * attempt)
    print(f"  [fail] could not download {name}")
    return None


def read_zip(path):
    """Read the CSV(s) inside a zip; finds the header row even if there are title lines above it."""
    frames = []
    with zipfile.ZipFile(path) as zf:
        for member in zf.namelist():
            if not member.lower().endswith(".csv"):
                continue
            text = zf.read(member).decode("utf-8-sig", errors="replace")
            lines = text.splitlines()
            hdr = next((i for i, l in enumerate(lines[:30])
                        if "FSA" in l.upper() and "DATE" in l.upper() and l.count(",") >= 3), None)
            if hdr is None:
                raise ValueError(f"No header row (with FSA and DATE) found in {member}")
            frames.append(pd.read_csv(io.StringIO(text), skiprows=hdr, dtype=str))
    if not frames:
        raise ValueError(f"No CSV inside {path}")
    return pd.concat(frames, ignore_index=True)


def standardise_columns(df):
    df = df.rename(columns={c: norm_col(c) for c in df.columns})
    rename = {}
    for canon, options in COLUMN_CANDIDATES.items():
        found = next((o for o in options if o in df.columns), None)
        if found is None:
            raise KeyError(f"No column found for {canon}. File has: {list(df.columns)}. "
                           f"Add the right name to COLUMN_CANDIDATES.")
        rename[found] = canon
    return df.rename(columns=rename)


def clean_and_engineer(df, ontario_holidays):
    df = standardise_columns(df)

    df["FSA"] = df["FSA"].astype(str).str.strip().str.upper()
    df["PRICE_PLAN"] = df["PRICE_PLAN"].astype(str).str.strip().str.upper()
    df["CUSTOMER_TYPE"] = df["CUSTOMER_TYPE"].astype(str).str.strip()

    df = df[df["PRICE_PLAN"].isin(KEEP_PRICE_PLANS)]      # Tiered / TOU only
    df = df[df["FSA"].str.match(FSA_RE)]                  # A1A only

    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
    for c in ("HOUR", "TOTAL_CONSUMPTION", "PREMISE_COUNT"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["DATE", "HOUR", "TOTAL_CONSUMPTION", "PREMISE_COUNT"])
    df = df[df["PREMISE_COUNT"] > 0]
    df["HOUR"] = df["HOUR"].astype(int)
    df = df[df["HOUR"].between(1, 24)]

    df["CONSUMPTION_PER_UNIT"] = df["TOTAL_CONSUMPTION"] / df["PREMISE_COUNT"]
    df["DAY_NAME"] = df["DATE"].dt.day_name()
    df["IS_WEEKDAY"] = (df["DATE"].dt.dayofweek < 5).astype(int)
    df["IS_HOLIDAY"] = df["DATE"].isin(ontario_holidays).astype(int)
    df["YEAR"] = df["DATE"].dt.year
    df["MONTH"] = df["DATE"].dt.month
    df["WEEK_OF_YEAR"] = df["DATE"].dt.isocalendar().week.astype(int)
    df["DAY_OF_YEAR"] = df["DATE"].dt.dayofyear
    df["REGION_LETTER"] = df["FSA"].str[0]                    # K, L, M, N, P
    df["IS_RURAL"] = (df["FSA"].str[1] == "0").astype(int)    # digit 0 = rural FSA
    return df


def load_centroids(path):
    c = pd.read_csv(path, dtype={"FSA": str})
    c.columns = [norm_col(x) for x in c.columns]
    c["FSA"] = c["FSA"].str.strip().str.upper()
    return c[["FSA", "LAT", "LON"]].drop_duplicates("FSA")


def load_temperature(path):
    t = pd.read_csv(path, dtype={"FSA": str})
    t.columns = [norm_col(x) for x in t.columns]
    t["FSA"] = t["FSA"].str.strip().str.upper()
    t["DATE"] = pd.to_datetime(t["DATE"])
    t["HOUR"] = t["HOUR"].astype(int)
    keep = ["FSA", "DATE", "HOUR"] + [c for c in WEATHER_COLS if c in t.columns]
    return t[keep].drop_duplicates(["FSA", "DATE", "HOUR"])


def split_and_append(df, out_dir):
    # Business day = weekday AND not an Ontario holiday
    biz = (df["IS_WEEKDAY"] == 1) & (df["IS_HOLIDAY"] == 0)
    cols = [c for c in OUTPUT_ORDER if c in df.columns]
    for (is_biz, hour), part in df.groupby([biz, df["HOUR"]]):
        prefix = "Business_Day" if is_biz else "Not_Business_Day"
        fpath = out_dir / f"{prefix}_{hour:02d}00.csv"
        part[cols].to_csv(fpath, mode="a", index=False,
                          header=not fpath.exists(), date_format="%Y-%m-%d")


# -------------------------------- main -------------------------------------- #
def main():
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    # clean slate so re-running never duplicates rows
    for old in list(out_dir.glob("Business_Day_*.csv")) + list(out_dir.glob("Not_Business_Day_*.csv")):
        old.unlink()

    years = range(START_YEAR, END_YEAR + 1)
    ontario_holidays = {pd.Timestamp(d) for d in holidays.Canada(subdiv="ON", years=years).keys()}
    print("Ontario holidays used (check these against the official list):")
    for d in sorted(ontario_holidays):
        print("  ", d.date())

    centroids = load_centroids(CENTROIDS_CSV) if CENTROIDS_CSV else None
    temps = load_temperature(TEMPERATURE_CSV) if TEMPERATURE_CSV else None

    total_rows, printed_values = 0, False
    with tempfile.TemporaryDirectory() as tmp:          # downloads are discarded automatically
        for y, m in month_range(START_YEAR, START_MONTH, END_YEAR, END_MONTH):
            print(f"{y}-{m:02d}")
            zpath = (Path(LOCAL_ZIP_DIR) / f"PUB_HourlyConsumptionByFSA_{y}{m:02d}_v1.zip"
                     if LOCAL_ZIP_DIR else download_zip(y, m, tmp))
            if not zpath or not Path(zpath).exists():
                continue

            raw = read_zip(zpath)
            print(f"  raw rows: {len(raw):,}")
            if PRINT_DISTINCT_VALUES and not printed_values:
                std = standardise_columns(raw)
                print("  PRICE_PLAN values  :", sorted(std["PRICE_PLAN"].astype(str).unique()))
                print("  CUSTOMER_TYPE values:", sorted(std["CUSTOMER_TYPE"].astype(str).unique()))
                printed_values = True

            df = clean_and_engineer(raw, ontario_holidays)

            if centroids is not None:
                df = df.merge(centroids, on="FSA", how="left")

            if temps is not None:
                df = df.merge(temps, on=["FSA", "DATE", "HOUR"], how="left")
                print(f"  rows missing temperature: {df['TEMPERATURE'].isna().mean():.1%}")
                df = df.dropna(subset=["TEMPERATURE"])
                df["HDD"] = (BASE_TEMP_C - df["TEMPERATURE"]).clip(lower=0)
                df["CDD"] = (df["TEMPERATURE"] - BASE_TEMP_C).clip(lower=0)

            print(f"  kept rows: {len(df):,}")
            total_rows += len(df)
            split_and_append(df, out_dir)

    n_files = len(list(out_dir.glob("*Business_Day_*.csv")))
    print(f"\nDone: {total_rows:,} rows in {n_files} files in '{out_dir}/'")


if __name__ == "__main__":
    main()


Ontario holidays used (check these against the official list):
   2022-01-01
   2022-01-03
   2022-02-21
   2022-04-15
   2022-05-23
   2022-07-01
   2022-09-05
   2022-10-10
   2022-12-25
   2022-12-26
   2023-01-01
   2023-01-02
   2023-02-20
   2023-04-07
   2023-05-22
   2023-07-01
   2023-09-04
   2023-10-09
   2023-12-25
   2023-12-26
2022-01
  raw rows: 1,297,616
  PRICE_PLAN values  : ['Retailer', 'TOU', 'Tiered']
  CUSTOMER_TYPE values: ['Residential', 'SGS <50kW']
  kept rows: 1,046,453
2022-02
  raw rows: 1,184,547
  kept rows: 947,568
2022-03
  raw rows: 1,309,480
  kept rows: 1,053,494
2022-04
  raw rows: 1,279,221
  kept rows: 1,024,765
2022-05
  raw rows: 1,327,765
  kept rows: 1,058,874
2022-06
  raw rows: 1,277,391
  kept rows: 1,024,690
2022-07
  raw rows: 1,322,617
  kept rows: 1,059,242
2022-08
  raw rows: 1,323,122
  kept rows: 1,059,089
2022-09
  raw rows: 1,273,002
  kept rows: 1,024,347
2022-10
  raw rows: 1,305,749
  kept rows: 1,057,101
2022-11
  raw rows: 1,2